# Notebook 02 — Prompt Analysis Agent Training (RoBERTa-base)
> Fine-tunes RoBERTa-base as a binary classifier on OCR-extracted insurance document text.
> GPU required (T4 on Colab). Expected training time: 30-60 min for 3 epochs.

In [ ]:
# Install dependencies (Colab)
import subprocess, sys
pkgs = ["transformers>=4.40.0", "datasets", "accelerate", "scikit-learn", "tqdm", "torch"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("Dependencies ready.")

In [ ]:
import os, sys, json, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ── Seeds (MANDATORY for reproducibility — cite in paper) ──────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────────
COLAB_BASE = "/content/drive/MyDrive/PAS"
LOCAL_BASE = str(Path(".").resolve().parent)
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = LOCAL_BASE

AGENTS_DIR  = os.path.join(LOCAL_BASE if not os.path.exists(COLAB_BASE) else COLAB_BASE, "../AGENTS").replace("\\\\", "/")
DATASET_DIR = os.path.join(BASE, "FINAL_BENCHMARK_DATASET")
CACHE_DIR   = os.path.join(BASE, "EXPERIMENT_CACHE")
MODELS_DIR  = os.path.join(BASE, "MODELS"); os.makedirs(MODELS_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(BASE, "RESULTS", "metrics"); os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
sys.path.insert(0, str(Path(".").resolve().parent / "AGENTS"))
from preprocessing_agent import PreprocessingAgent

agent = PreprocessingAgent(cache_dir=CACHE_DIR)
train_df, val_df, test_df = agent.load_splits()

# Determine text column
TEXT_COL = next((c for c in ["text", "ocr_text", "prompt"] if c in train_df.columns), None)
print(f"Using text column: {TEXT_COL}")

train_texts = train_df[TEXT_COL].fillna("").tolist()
train_labels = train_df["label_int"].tolist()
val_texts   = val_df[TEXT_COL].fillna("").tolist()
val_labels   = val_df["label_int"].tolist()
test_texts  = test_df[TEXT_COL].fillna("").tolist()
test_labels  = test_df["label_int"].tolist()
print(f"Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}")

In [ ]:
from transformers import AutoTokenizer
from preprocessing_agent import smart_truncate

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Applying smart truncation (head128+mid256+tail128) for long documents...")
train_texts_trunc = [smart_truncate(t, tokenizer) for t in train_texts]
val_texts_trunc   = [smart_truncate(t, tokenizer) for t in val_texts]
test_texts_trunc  = [smart_truncate(t, tokenizer) for t in test_texts]

train_enc = tokenizer(train_texts_trunc, padding=True, truncation=True, max_length=512, return_tensors="pt")
val_enc   = tokenizer(val_texts_trunc,   padding=True, truncation=True, max_length=512, return_tensors="pt")
test_enc  = tokenizer(test_texts_trunc,  padding=True, truncation=True, max_length=512, return_tensors="pt")
print("Tokenization complete.")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from preprocessing_agent import PreprocessingAgent

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()} | {"labels": torch.tensor(self.labels[idx])}

train_dataset = TextDataset(train_enc, train_labels)
val_dataset   = TextDataset(val_enc,   val_labels)
test_dataset  = TextDataset(test_enc,  test_labels)

# Class weights for imbalance
cw = PreprocessingAgent.compute_class_weights(np.array(train_labels))
imbalance_ratio = cw[1] / cw[0]
print(f"Imbalance ratio benign:malicious = {imbalance_ratio:.2f}")
print("Using class weights: Yes" if imbalance_ratio > 1.5 else "Using class weights: No (balanced enough)")

In [ ]:
from transformers import (
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1_macro":  f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "roc_auc":   roc_auc_score(labels, probs),
        "fnr":       (labels[(labels==1) & (preds==0)].sum() / max(1, (labels==1).sum())),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

ROBERTA_DIR = os.path.join(MODELS_DIR, "roberta_classifier")

training_args = TrainingArguments(
    output_dir=ROBERTA_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=(DEVICE == "cuda"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt

# Evaluate on test set
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_probs = torch.softmax(torch.tensor(test_results.predictions), dim=-1)[:, 1].numpy()

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(classification_report(test_labels, test_preds, target_names=["benign", "malicious"]))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["benign", "malicious"],
            yticklabels=["benign", "malicious"])
plt.title("RoBERTa Test Confusion Matrix", fontweight="bold")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.savefig(os.path.join(BASE, "RESULTS", "confusion_matrices", "02_roberta_cm.png"),
            bbox_inches="tight", dpi=150)
plt.show()

# Save predictions CSV
pred_df = pd.DataFrame({
    "sample_id": test_df["sample_id"].tolist(),
    "true_label": test_labels,
    "predicted_label": test_preds,
    "malicious_probability": test_probs,
})
if "attack_family" in test_df.columns:
    pred_df["attack_family"] = test_df["attack_family"].tolist()
pred_df.to_csv(os.path.join(RESULTS_DIR, "prompt_agent_predictions.csv"), index=False)
print(f"Predictions saved: {len(pred_df):,} rows")

# Save model
trainer.save_model(ROBERTA_DIR)
tokenizer.save_pretrained(ROBERTA_DIR)
print(f"Model saved to: {ROBERTA_DIR}")